> Historical tuning notebook. See 04_t4_video_gradio.ipynb for the video upload demo.

# C389 CA1 — T4: Wait or cross

**Start here.** This notebook uses your existing improved `artifacts/L3/best.pt`.
It adds ByteTrack identities, a waiting polygon, a crossing line, waiting episodes,
annotated video, and manual evaluation. Gradio belongs in a separate demo notebook.

## How to run
1. Upload this notebook to `c389 ca1 final (improved model)/notebook/` and open in Colab.
2. Select a GPU runtime. Run the setup and engine cells in order.
3. In **Your files**, set the real video name. Put the video in `data/clips/` under that Drive folder.
4. Run the frame preview. In **Your site**, enter the actual zone corners, line, capture date and camera position.
5. Inspect the marked first/middle/last frames. Confirm the geometry and video timing in the next cell.
6. Run tracking, analysis and the annotated replay. Check the IDs by eye.
7. Annotate all real people independently, map them to the visible IDs, then enable evaluation.
8. Tune using tuning clips. Freeze settings before using different held-out test clips.

**No real video, site geometry or ground truth is supplied.** The notebook deliberately stops
where those are required. Initial thresholds are trial values; they are not an optimal result
or a prediction of your grade. No additional training is performed.

## Operational definitions implemented
- A foot point is the bottom centre of a detected box; its recent median reduces jitter.
- A possible wait needs a full motion window of observations inside the waiting polygon.
- Stillness uses maximum displacement from the start of that window, divided by median
  box height and elapsed time. Units are body heights per second, **not metres per second**.
- A wait qualifies after `min_wait_seconds` of continuous observed stillness.
  Zone dwell while moving does not qualify. Short observation gaps up to the configured limit
  are assumed continuous; larger gaps split episodes and reset motion/crossing memory.
- Crossing means passing the **finite** marked segment, with a dead band and sustained side
  confirmation. Place it where passing the line actually means entering/crossing your walkway.
- Each eligible ID contributes at most once to each total. Waited and crossed totals overlap;
  the four outcome categories are exclusive. One real person can still acquire several IDs.
- A wait starting at a track/gap boundary or ending at a track/gap boundary is incomplete:
  report its duration as **at least** the observed duration. Unknown time is never joined across a large gap.

These are inspectable rules for your team to test and justify. Pixel geometry assumes one fixed
camera position. Body-height normalisation reduces scale differences but does not calibrate
perspective or eliminate box jitter. Gradual movement below the threshold can still be called still.

## Relationship to the supplied lessons
Uses the tracking and foot-point approach in `L5B_codes_tracking_best.ipynb` and the area/line
geometry from `L5B_marking_regions.ipynb`. The stock-model notebook is a comparison reference.
T4 does not need the new-class lab or T5's measured ground distance. The improved directional
model must still be checked on stationary/front-facing people, which its training data may miss.

Technical reference: [Ultralytics tracking documentation](https://docs.ultralytics.com/modes/track/).
The install remains pinned to the supplied lessons' version for consistency.
This is implementation/reference code; write your own report and individual worksheet explanations
from your actual work, as required by the CA1 brief.

## 1. Setup
This installs the same Ultralytics version used by your lessons. It needs internet on Colab.
Every video gets a fresh model/tracker instance, so IDs do not continue from another clip.

In [ ]:
import sys
import subprocess
try:
    import google.colab
    RUNNING_ON_COLAB = True
except ImportError:
    RUNNING_ON_COLAB = False

ULTRALYTICS_VERSION = "8.4.120"
if RUNNING_ON_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"ultralytics=={ULTRALYTICS_VERSION}", "lap", "pandas", "matplotlib"], check=True)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
import ultralytics
from IPython.display import display, Video
from datetime import datetime, timezone
from uuid import uuid4
from PIL import Image
if ultralytics.__version__ != ULTRALYTICS_VERSION:
    raise RuntimeError(f"Expected Ultralytics {ULTRALYTICS_VERSION}; install that version and restart the runtime.")
print("Ultralytics:", ultralytics.__version__, "| OpenCV:", cv2.__version__)


## 2. Shared settings and file helpers

In [ ]:
# Engine section: configuration
from collections import Counter
from dataclasses import asdict, dataclass, replace
from pathlib import Path
import csv
import hashlib
import json
import math
import statistics
import time


@dataclass(frozen=True)
class Rules:
    min_wait_seconds: float = 3.0
    motion_window_seconds: float = 0.75
    smoothing_seconds: float = 0.20
    still_body_heights_per_second: float = 0.06
    max_observation_gap_seconds: float = 0.25
    min_track_seconds: float = 0.5
    min_track_observations: int = 5
    line_margin_body_heights: float = 0.03
    line_confirm_seconds: float = 0.15

    def validate(self):
        for key, value in asdict(self).items():
            if not math.isfinite(value) or value <= 0:
                raise ValueError(f'{key} must be finite and greater than zero.')
        if self.min_wait_seconds < self.motion_window_seconds:
            raise ValueError('Minimum wait must be at least the motion window.')
        if self.min_track_observations != int(self.min_track_observations):
            raise ValueError('min_track_observations must be a whole number.')


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def save_json(path, value):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')


def save_csv(path, rows, fields=None):
    rows = list(rows)
    fields = fields or (list(rows[0]) if rows else [])
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: json.dumps(v) if isinstance(v, (list, dict)) else v
                             for k, v in row.items()})


## 3. Area and finite-line geometry

In [ ]:
# Engine section: geometry
def point_in_polygon(point, polygon):
    """Boundary is included. Polygon is in frame pixels, ordered round its edge."""
    x, y = point
    inside = False
    for a, b in zip(polygon, polygon[1:] + polygon[:1]):
        ax, ay = a
        bx, by = b
        cross = (bx - ax) * (y - ay) - (by - ay) * (x - ax)
        if abs(cross) < 1e-7 and min(ax, bx) <= x <= max(ax, bx) and min(ay, by) <= y <= max(ay, by):
            return True
        if (ay > y) != (by > y):
            hit = ax + (y - ay) * (bx - ax) / (by - ay)
            if x < hit:
                inside = not inside
    return inside


def line_distance(point, line):
    (ax, ay), (bx, by) = line
    length = math.hypot(bx - ax, by - ay)
    if not length:
        raise ValueError('Crossing-line endpoints must differ.')
    return ((bx - ax) * (point[1] - ay) - (by - ay) * (point[0] - ax)) / length


def finite_line_crossing(a, b, line):
    """Return interpolation fraction when the motion crosses the FINITE segment."""
    da, db = line_distance(a, line), line_distance(b, line)
    if da * db > 0 or abs(da - db) < 1e-12:
        return None
    fraction = da / (da - db)
    px, py = a[0] + fraction * (b[0] - a[0]), a[1] + fraction * (b[1] - a[1])
    (ax, ay), (bx, by) = line
    projection = ((px - ax) * (bx - ax) + (py - ay) * (by - ay)) / ((bx - ax)**2 + (by - ay)**2)
    return fraction if 0 <= fraction <= 1 and 0 <= projection <= 1 else None


def validate_site(site, width, height):
    if site.get('frame_width') != width or site.get('frame_height') != height:
        raise ValueError('Site resolution differs from the video. Mark this camera setup again.')
    area, line = site.get('waiting_area'), site.get('crossing_line')
    if not area or len(area) < 3 or not line or len(line) != 2:
        raise ValueError('Set waiting_area corners and two crossing_line endpoints from your frame.')
    for point in list(area) + list(line):
        if len(point) != 2 or not all(math.isfinite(v) for v in point):
            raise ValueError('Every site point must contain two finite coordinates.')
        if not (0 <= point[0] < width and 0 <= point[1] < height):
            raise ValueError(f'Point {point} lies outside this frame.')
    signed_area = sum(a[0]*b[1] - b[0]*a[1] for a,b in zip(area, area[1:] + area[:1]))
    if abs(signed_area) < 1:
        raise ValueError('Waiting area has zero area or invalid corner order.')
    line_distance((0, 0), line)
    if site.get('crossing_direction') not in ('both', 'negative_to_positive', 'positive_to_negative'):
        raise ValueError('Choose a crossing_direction: both, negative_to_positive, positive_to_negative.')
    if not site.get('recorded_on') or not site.get('camera_position'):
        raise ValueError('Record the capture date and camera position in SITE.')


## 4. Waiting episodes from observed motion

In [ ]:
# Engine section: waiting_analysis
def _prepare_observations(observations, rules):
    """Causal median smoothing, reset at gaps; no interpolation across missing spans."""
    prepared, history, segment = [], [], 0
    previous = None
    for original in observations:
        row = dict(original)
        t = float(row['seconds'])
        if not math.isfinite(t) or t < 0:
            raise ValueError('Track timestamps must be finite and nonnegative.')
        if previous is not None and t <= previous:
            raise ValueError('Each track must have strictly increasing timestamps.')
        if previous is not None and t - previous > rules.max_observation_gap_seconds + 1e-9:
            history = []
            segment += 1
        box = row['box']
        if len(box) != 4 or not all(math.isfinite(v) for v in box) or box[3] <= box[1] or box[2] <= box[0]:
            raise ValueError('Each observation needs a finite positive-size box.')
        if len(row['foot']) != 2 or not all(math.isfinite(v) for v in row['foot']):
            raise ValueError('Each observation needs a finite foot point.')
        history.append(row)
        history = [r for r in history if t - r['seconds'] <= rules.smoothing_seconds + 1e-9]
        row['smooth_foot'] = [statistics.median(r['foot'][axis] for r in history) for axis in (0,1)]
        row['height'] = box[3] - box[1]
        row['segment'] = segment
        prepared.append(row)
        previous = t
    return prepared


def waiting_episodes(prepared, area, rules):
    """Observe continuous stillness INSIDE the zone; qualify only at minimum duration.

    Motion is the largest displacement from the window's first smoothed foot,
    divided by median box height and elapsed seconds. It is an image-space cue,
    not physical speed. Reversing back to the start cannot erase earlier motion.
    """
    episodes, timeline, window = [], [], []
    active = None
    segment_first = None

    def finish(reason):
        nonlocal active
        if active is not None:
            active['seconds'] = round(active['end_seconds'] - active['start_seconds'], 6)
            active['qualified'] = active['seconds'] + 1e-9 >= rules.min_wait_seconds
            active['end_reason'] = reason
            active['right_censored'] = reason in ('observation_gap', 'track_end')
            active['incomplete'] = active['left_censored'] or active['right_censored']
            episodes.append(active)
            active = None

    for i, row in enumerate(prepared):
        t = row['seconds']
        if i == 0 or row['segment'] != prepared[i-1]['segment']:
            finish('observation_gap')
            window = []
            segment_first = t
        inside = point_in_polygon(row['smooth_foot'], area)
        if not inside:
            finish('zone_exit')
            window = []
            state, motion = 'outside_waiting_area', None
        else:
            window.append(row)
            # Retain a point at or just before the beginning of the window.
            while len(window) > 1 and window[1]['seconds'] <= t - rules.motion_window_seconds:
                window.pop(0)
            elapsed = t - window[0]['seconds']
            if elapsed + 1e-9 < rules.motion_window_seconds:
                state, motion = 'observing', None
            else:
                origin = window[0]['smooth_foot']
                extent = max(math.dist(origin, r['smooth_foot']) for r in window)
                motion = extent / statistics.median(r['height'] for r in window) / elapsed
                if motion <= rules.still_body_heights_per_second:
                    if active is None:
                        active = {'start_seconds': window[0]['seconds'], 'end_seconds': t,
                                  'left_censored': window[0]['seconds'] <= segment_first + 1e-9}
                    active['end_seconds'] = t
                    state = ('waiting' if t - active['start_seconds'] + 1e-9 >= rules.min_wait_seconds
                             else 'possible_wait')
                else:
                    finish('movement')
                    state = 'moving_in_waiting_area'
        timeline.append({'frame': row['frame'], 'seconds': t, 'state': state,
                         'motion_body_heights_per_second': motion,
                         'observed_wait_seconds': round(t - active['start_seconds'], 6) if active else 0.0,
                         'smooth_foot': row['smooth_foot']})
    finish('track_end')
    return episodes, timeline


## 5. Crossing events and per-person outcomes

In [ ]:
# Engine section: crossing_analysis
def crossing_events(prepared, site, rules):
    """Hysteresis and side confirmation reject jitter; repeated events stay visible."""
    events = []
    stable_side, anchor, candidate = None, None, None
    previous_segment = None
    for row in prepared:
        if row['segment'] != previous_segment:
            stable_side, anchor, candidate = None, None, None
            previous_segment = row['segment']
        distance = line_distance(row['smooth_foot'], site['crossing_line'])
        margin = rules.line_margin_body_heights * row['height']
        side = 1 if distance > margin else -1 if distance < -margin else 0
        if side == 0:
            candidate = None
            continue
        if side == stable_side:
            anchor, candidate = row, None
            continue
        if candidate is None or candidate['side'] != side:
            candidate = {'side': side, 'row': row}
        if row['seconds'] - candidate['row']['seconds'] + 1e-9 < rules.line_confirm_seconds:
            continue
        if stable_side is not None:
            fraction = finite_line_crossing(anchor['smooth_foot'], candidate['row']['smooth_foot'],
                                            site['crossing_line'])
            direction = 'negative_to_positive' if stable_side == -1 else 'positive_to_negative'
            if fraction is not None and site['crossing_direction'] in ('both', direction):
                event_t = anchor['seconds'] + fraction * (candidate['row']['seconds'] - anchor['seconds'])
                dx = candidate['row']['smooth_foot'][0] - anchor['smooth_foot'][0]
                events.append({'seconds': round(event_t, 6), 'confirmed_seconds': row['seconds'],
                               'direction': direction,
                               'screen_direction': 'right' if dx > 1e-6 else 'left' if dx < -1e-6 else 'vertical'})
        stable_side, anchor, candidate = side, row, None
    return events


def analyze_tracks(tracks, site, rules):
    rules.validate()
    people, all_episodes, all_events, timelines = [], [], [], {}
    for track_id, observations in sorted(tracks.items(), key=lambda item: int(item[0])):
        if not observations:
            continue
        prepared = _prepare_observations(observations, rules)
        episodes, timeline = waiting_episodes(prepared, site['waiting_area'], rules)
        events = crossing_events(prepared, site, rules)
        track_id = int(track_id)
        for e in episodes:
            e['track_id'] = track_id
        for e in events:
            e['track_id'] = track_id
        waits = [e for e in episodes if e['qualified']]
        eligible = (len(prepared) >= rules.min_track_observations and
                    prepared[-1]['seconds'] - prepared[0]['seconds'] >= rules.min_track_seconds)
        transitions = [(w,e) for w in waits for e in events
                       if w['start_seconds'] + rules.min_wait_seconds <= e['seconds'] + 1e-9]
        if transitions:
            outcome = 'waited_then_crossed'
        elif waits:
            outcome = 'waited_without_later_crossing'
        elif events:
            outcome = 'crossed_without_observed_wait'
        else:
            outcome = 'unresolved'
        flags = []
        if not eligible:
            flags.append('short_track_excluded_from_counts')
        if prepared[-1]['segment']:
            flags.append('observation_gaps')
        if point_in_polygon(prepared[0]['smooth_foot'], site['waiting_area']):
            flags.append('track_begins_inside_waiting_area_prior_behaviour_unknown')
        if any(w['incomplete'] for w in waits):
            flags.append('waiting_duration_is_lower_bound')
        if len(events) > 1:
            flags.append('multiple_crossings_counted_once')
        if waits and events and not transitions:
            flags.append('crossing_precedes_observed_wait')
        majority = Counter(r.get('class_name','unknown') for r in observations).most_common(1)[0][0]
        people.append({'track_id': track_id, 'eligible': eligible, 'outcome': outcome,
                       'waited': bool(waits), 'crossed': bool(events),
                       'observed_wait_seconds': round(sum(w['seconds'] for w in waits), 6),
                       'wait_episode_count': len(waits),
                       'duration_incomplete': any(w['incomplete'] for w in waits),
                       'first_crossing_seconds': events[0]['seconds'] if events else None,
                       'first_seen_seconds': prepared[0]['seconds'], 'last_seen_seconds': prepared[-1]['seconds'],
                       'observations': len(prepared), 'majority_detector_class': majority,
                       'flags': flags})
        timelines[track_id] = timeline
        all_episodes.extend(episodes)
        all_events.extend(events)
    included = [r for r in people if r['eligible']]
    outcomes = Counter(r['outcome'] for r in included)
    summary = {'raw_track_ids': len(people), 'eligible_track_ids': len(included),
               'excluded_short_track_ids': len(people) - len(included),
               'waited_track_ids': sum(r['waited'] for r in included),
               'crossed_track_ids': sum(r['crossed'] for r in included),
               'exclusive_outcomes': {k: outcomes[k] for k in (
                   'waited_then_crossed','waited_without_later_crossing',
                   'crossed_without_observed_wait','unresolved')},
               'count_unit': 'eligible tracker identity; fragmentation can overcount real people',
               'note': 'Waited and crossed overlap. Exclusive outcomes sum to eligible identities.'}
    return {'summary': summary, 'people': people, 'wait_episodes': all_episodes,
            'crossings': all_events, 'timelines': timelines}


## 6. Honest evaluation against manual ground truth

In [ ]:
# Engine section: evaluation
def parse_bool(value, field):
    if str(value).strip().lower() in ('1','true','yes'):
        return True
    if str(value).strip().lower() in ('0','false','no'):
        return False
    raise ValueError(f'{field}: use true or false; do not leave an unlabelled value.')


def evaluate_people(result, truth_rows, mapping_rows):
    """Manual full-population matching, including missed humans and unmatched tracks.

    Raw system count errors retain fragmentation. Person-grouped classification
    is labelled separately; it never substitutes grouped counts for raw counts.
    Exact duration MAE excludes censored/fragmented cases and reports coverage.
    """
    if not truth_rows:
        raise ValueError('Ground truth is empty: annotate every visible relevant person first.')
    truth = {}
    for raw in truth_rows:
        human = str(raw['human_id']).strip()
        if not human or human in truth:
            raise ValueError('Ground-truth human_id values must be nonempty and unique.')
        waited = parse_bool(raw['waited'], 'waited')
        crossed = parse_bool(raw['crossed'], 'crossed')
        complete = parse_bool(raw['duration_complete'], 'duration_complete')
        duration = None if str(raw.get('wait_seconds','')).strip() == '' else float(raw['wait_seconds'])
        if duration is not None and (not math.isfinite(duration) or duration < 0):
            raise ValueError('wait_seconds must be nonnegative and finite.')
        if waited and complete and duration is None:
            raise ValueError('A complete true wait requires hand-measured wait_seconds.')
        truth[human] = {'waited': waited, 'crossed': crossed,
                        'wait_seconds': duration, 'duration_complete': complete}
    predicted = {int(r['track_id']): r for r in result['people'] if r['eligible']}
    all_ids = {int(r['track_id']) for r in result['people']}
    mapping = {human: [] for human in truth}
    assigned = set()
    for row in mapping_rows:
        human = str(row['human_id']).strip()
        if human not in truth:
            raise ValueError(f'Unknown human_id in mapping: {human}')
        raw_id = str(row.get('track_id','')).strip()
        if not raw_id:
            continue
        tid = int(raw_id)
        if tid not in all_ids or tid in assigned:
            raise ValueError('Mapped track IDs must exist and may belong to only one human.')
        assigned.add(tid)
        mapping[human].append(tid)
    rows, duration_errors, exclusions = [], [], []
    for human, target in truth.items():
        matched = [predicted[t] for t in mapping[human] if t in predicted]
        pw, pc = any(r['waited'] for r in matched), any(r['crossed'] for r in matched)
        exact = (target['waited'] and target['duration_complete'] and len(matched) == 1 and
                 matched[0]['waited'] and not matched[0]['duration_incomplete'])
        error = abs(matched[0]['observed_wait_seconds'] - target['wait_seconds']) if exact else None
        if exact:
            duration_errors.append(error)
        elif target['waited']:
            exclusions.append({'human_id': human, 'reason': 'missed wait, incomplete duration, or fragmented mapping'})
        rows.append({'human_id': human, 'mapped_track_ids': mapping[human],
                     'detected': bool(matched), 'truth_waited': target['waited'], 'predicted_waited': pw,
                     'truth_crossed': target['crossed'], 'predicted_crossed': pc,
                     'classification_correct': bool(matched) and pw == target['waited'] and pc == target['crossed'],
                     'duration_absolute_error_seconds': error})
    unmatched = [r for t,r in predicted.items() if t not in assigned]
    metrics = {'ground_truth_people': len(truth),
               'people_with_eligible_track': sum(r['detected'] for r in rows),
               'unmatched_eligible_track_ids': [r['track_id'] for r in unmatched],
               'humans_with_multiple_track_ids': sum(len(ids) > 1 for ids in mapping.values()),
               'person_grouped_classification_accuracy': sum(r['classification_correct'] for r in rows) / len(rows),
               'duration_mae_seconds': statistics.mean(duration_errors) if duration_errors else None,
               'duration_evaluated_people': len(duration_errors),
               'ground_truth_waiting_people': sum(t['waited'] for t in truth.values()),
               'duration_exclusions': exclusions}
    for name in ('waited','crossed'):
        true_count = sum(t[name] for t in truth.values())
        raw_count = result['summary'][f'{name}_track_ids']
        tp = sum(r[f'truth_{name}'] and r[f'predicted_{name}'] for r in rows)
        fp = sum(not r[f'truth_{name}'] and r[f'predicted_{name}'] for r in rows) + sum(r[name] for r in unmatched)
        fn = sum(r[f'truth_{name}'] and not r[f'predicted_{name}'] for r in rows)
        metrics[f'{name}_truth_count'] = true_count
        metrics[f'{name}_raw_predicted_count'] = raw_count
        metrics[f'{name}_raw_count_absolute_error'] = abs(raw_count - true_count)
        metrics[f'{name}_person_grouped_precision'] = tp / (tp+fp) if tp+fp else None
        metrics[f'{name}_person_grouped_recall'] = tp / (tp+fn) if tp+fn else None
        metrics[f'{name}_person_grouped_f1'] = 2*tp / (2*tp+fp+fn) if 2*tp+fp+fn else None
    return {'metrics': metrics, 'people': rows}


## 7. Video and tracking functions

In [ ]:
# Engine section: video_tracking
def video_info(path):
    import cv2
    cap = cv2.VideoCapture(str(path))
    try:
        if not cap.isOpened():
            raise ValueError(f'Cannot open video: {path}')
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if not math.isfinite(fps) or fps <= 0 or frames <= 0 or min(width,height) <= 0:
            raise ValueError('Video has invalid FPS, frame count, or resolution. Export constant-frame-rate MP4.')
        return {'fps': fps, 'frames': frames, 'width': width, 'height': height,
                'duration_seconds': frames/fps,
                'timing_basis': 'frame index / container FPS; requires constant frame rate'}
    finally:
        cap.release()


def read_video_frame(path, frame=0):
    import cv2
    cap = cv2.VideoCapture(str(path))
    try:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame)
        ok, image = cap.read()
        if not ok:
            raise ValueError(f'Cannot decode frame {frame}.')
        return image
    finally:
        cap.release()


def track_video(clip_path, model_path, tracker_path, confidence=0.10, image_size=640, progress=None):
    import cv2
    from ultralytics import YOLO
    if not Path(model_path).is_file():
        raise FileNotFoundError('Trained best.pt is missing. No automatic stock/sample fallback is used.')
    info = video_info(clip_path)
    model = YOLO(str(model_path))  # Fresh object: identities cannot leak from another clip.
    names = model.names
    class_ids = [int(i) for i,name in names.items() if name in ('person_left','person_right','person')]
    if not class_ids:
        raise ValueError(f'Model contains no recognised person classes: {names}')
    cap, tracks, decoded, started = cv2.VideoCapture(str(clip_path)), {}, 0, time.perf_counter()
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            result = model.track(frame, persist=True, tracker=str(tracker_path), classes=class_ids,
                                 conf=confidence, imgsz=image_size, verbose=False)[0]
            boxes = result.boxes
            if boxes is not None and boxes.id is not None:
                for box,tid,cid,conf in zip(boxes.xyxy.tolist(), boxes.id.int().tolist(),
                                           boxes.cls.int().tolist(), boxes.conf.tolist()):
                    left,top,right,bottom = box
                    tracks.setdefault(int(tid), []).append({'frame': decoded, 'seconds': decoded/info['fps'],
                        'box': box, 'foot': [(left+right)/2, bottom],
                        'class_name': names[int(cid)], 'confidence': float(conf)})
            decoded += 1
            if progress and decoded % max(1,round(info['fps']*5)) == 0:
                progress(decoded, info['frames'])
    finally:
        cap.release()
    if decoded != info['frames']:
        raise RuntimeError(f'Decoded {decoded}/{info["frames"]} frames. Repair/re-export the video before evaluation.')
    elapsed = time.perf_counter() - started
    return {'clip': str(clip_path), 'model': str(model_path), 'clip_info': info, 'tracks': tracks,
            'class_names': names, 'decoded_frames': decoded, 'processing_seconds': elapsed,
            'processing_fps': decoded / elapsed if elapsed else None}


## 8. Annotated video function

In [ ]:
# Engine section: rendering
def render_video(clip_path, tracked, result, site, rules, output_path):
    """Offline annotated replay. Uses past observations only for displayed states."""
    import cv2
    import numpy as np
    info = tracked['clip_info']
    frames = {}
    for tid, observations in tracked['tracks'].items():
        tid = int(tid)
        states = {r['frame']: r for r in result['timelines'][tid]}
        for obs in observations:
            frames.setdefault(obs['frame'], []).append((tid, obs, states[obs['frame']]))
    eligible = {r['track_id'] for r in result['people'] if r['eligible']}
    cap = cv2.VideoCapture(str(clip_path))
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*'mp4v'),
                             info['fps'], (info['width'],info['height']))
    if not writer.isOpened():
        cap.release()
        raise RuntimeError('Video encoder could not open the output file.')
    i = 0
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            area = np.array(site['waiting_area'],dtype=np.int32)
            overlay = frame.copy()
            cv2.fillPoly(overlay,[area],(20,170,230))
            frame = cv2.addWeighted(overlay,0.18,frame,0.82,0)
            cv2.polylines(frame,[area],True,(20,170,230),2)
            a,b = [tuple(map(int,p)) for p in site['crossing_line']]
            cv2.line(frame,a,b,(255,150,30),3)
            for tid,obs,state in frames.get(i,[]):
                x1,y1,x2,y2 = map(int,obs['box'])
                colour = (30,210,30) if state['state']=='waiting' else (30,170,255)
                if tid not in eligible:
                    colour = (150,150,150)
                cv2.rectangle(frame,(x1,y1),(x2,y2),colour,2)
                label = (
    f"ID {tid} {obs['class_name']} "
    f"{state['state']} "
    f"{state['observed_wait_seconds']:.1f}s"
)
                cv2.putText(frame,label,(max(0,x1),max(18,y1-8)),cv2.FONT_HERSHEY_SIMPLEX,0.48,colour,2)
                cv2.circle(frame,tuple(map(int,state['smooth_foot'])),4,colour,-1)
            now = i / info['fps']
            waited = {w['track_id'] for w in result['wait_episodes'] if w['qualified'] and
                      w['track_id'] in eligible and w['start_seconds'] + rules.min_wait_seconds <= now}
            crossed = {e['track_id'] for e in result['crossings'] if e['track_id'] in eligible and e['confirmed_seconds'] <= now}
            cv2.rectangle(frame,(0,0),(info['width'],34),(20,20,20),-1)
            cv2.putText(frame,f'{now:.1f}s | waited IDs {len(waited)} | crossed IDs {len(crossed)}',
                        (8,24),cv2.FONT_HERSHEY_SIMPLEX,0.60,(255,255,255),1)
            writer.write(frame)
            i += 1
    finally:
        cap.release()
        writer.release()
    if i != tracked['decoded_frames']:
        raise RuntimeError('Annotated replay decoded a different number of frames.')
    return str(output_path)


## 9. Your files — edit this cell
Default paths point to your improved-model folder. Use a **new real video** for final testing.
Use a constant-frame-rate recording/export: timing below is frame index divided by FPS.
Phone variable-frame-rate footage should be converted to constant frame rate before labelling.
For example, Colab's `ffmpeg` can export a separate file with `-vf fps=30 -fps_mode cfr`;
do not overwrite your original, and measure ground truth on the same exported file used here.

For the first run use `PHASE = "tuning"`. For a held-out test use `"test"` and a saved frozen-settings file.

In [ ]:
if RUNNING_ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/c389 ca1 final (improved model)")
# Outside Colab, replace PROJECT_ROOT with your local project directory.
CLIP_NAME = "tuning_video1.MOV"
PHASE = "tuning"                          # "tuning" or "test"
FROZEN_SETTINGS_PATH = None                # For test: full path to frozen_settings_....json

MODEL_PATH = PROJECT_ROOT / "artifacts" / "L3" / "best.pt"
CLIP_PATH = PROJECT_ROOT / "data" / "clips" / CLIP_NAME
if PHASE not in ("tuning", "test"):
    raise ValueError("PHASE must be tuning or test.")
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"Put your trained best.pt here: {MODEL_PATH}")
if not CLIP_PATH.is_file():
    raise FileNotFoundError(f"Upload your real video here: {CLIP_PATH}")

CLIP_INFO = video_info(CLIP_PATH)
print("Model:", MODEL_PATH)
print("Video:", CLIP_PATH)
print(json.dumps(CLIP_INFO, indent=2))
FIRST_FRAME = read_video_frame(CLIP_PATH)
fig, ax = plt.subplots(figsize=(14,7))
ax.imshow(cv2.cvtColor(FIRST_FRAME,cv2.COLOR_BGR2RGB))
ax.set_xticks(np.linspace(0,CLIP_INFO['width']-1,11).astype(int))
ax.set_yticks(np.linspace(0,CLIP_INFO['height']-1,11).astype(int))
ax.grid(alpha=0.6)
ax.set_title("Read your ground-zone corners and crossing-line endpoints from these pixel coordinates")
plt.show()


## 10. Your site and trial thresholds — edit this cell
Enter polygon corners **around the edge**, using the preview. Avoid a self-intersecting shape.
Use the actual ground waiting area, not a rectangle around somebody's body.
The single crossing line belongs to T4's crossing decision; it is not a T5 speed measurement.

Line orientation determines the names of its two sides. Begin with `"both"`; use a direction
filter only if your agreed scope requires it. In test mode all geometry, rules and tracker
settings are loaded from the frozen file. If the camera moved, this is a different site setup.

The motion threshold `0.06` means about 6 pixels/second for a 100-pixel-tall box.
Test this on a stationary person and a slow walker at near and far positions.
`DETECTION_CONFIDENCE = 0.10` lets ByteTrack see the lower-confidence detections used in its
second association stage. It is separate from your image-service threshold of 0.40.

In [ ]:
SITE = {
    "site_id": "walkway_camera_01",
    "frame_width": CLIP_INFO["width"],
    "frame_height": CLIP_INFO["height"],
    "recorded_on": 2026-10-22,                    # Example format: "2026-11-03"; enter actual date
    "camera_position": "Camera looking diagonally across the zebra crossing, with both pavement approaches visible",                # Describe actual mounting position and direction
  "waiting_area": [
    [680, 290],   # Top-left
    [835, 290],   # Top-right
    [835, 410],   # Bottom-right
    [650, 410],   # Bottom-left
],                  # e.g. [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
    "crossing_line": [
    [420, 315],   # Upper endpoint
    [420, 400],   # Lower endpoint
],                  # [[x1,y1],[x2,y2]] inside the frame
    "crossing_direction": "both",
}
RULES = Rules(
    min_wait_seconds=1.0,
    motion_window_seconds=0.75,
    smoothing_seconds=0.20,
    still_body_heights_per_second=0.06,
    max_observation_gap_seconds=0.25,
    min_track_seconds=0.5,
    min_track_observations=5,
    line_margin_body_heights=0.03,
    line_confirm_seconds=0.15,
)
DETECTION_CONFIDENCE = 0.10
IMAGE_SIZE = 640
TRACKER_CHANGES = dict(track_high_thresh=0.25, track_low_thresh=0.10,
                       new_track_thresh=0.25, track_buffer=30, match_thresh=0.80)

from ultralytics.utils import ROOT as ULTRALYTICS_ROOT
default_tracker = Path(ULTRALYTICS_ROOT) / "cfg" / "trackers" / "bytetrack.yaml"
TRACKER_SETTINGS = yaml.safe_load(default_tracker.read_text())
unknown = set(TRACKER_CHANGES) - set(TRACKER_SETTINGS)
if unknown:
    raise ValueError(f"Tracker keys absent from this installed version: {unknown}")
TRACKER_SETTINGS.update(TRACKER_CHANGES)
FROZEN = None
if PHASE == "test":
    if not FROZEN_SETTINGS_PATH or not Path(FROZEN_SETTINGS_PATH).is_file():
        raise ValueError("Test mode requires the frozen-settings JSON from your tuning run.")
    FROZEN = json.loads(Path(FROZEN_SETTINGS_PATH).read_text())
    if FROZEN['ultralytics_version'] != ULTRALYTICS_VERSION:
        raise ValueError('Frozen settings used a different Ultralytics version.')
    SITE = FROZEN["site"]
    RULES = Rules(**FROZEN["rules"])
    TRACKER_SETTINGS = FROZEN["tracker_settings"]
    DETECTION_CONFIDENCE = FROZEN["detection_confidence"]
    IMAGE_SIZE = FROZEN["image_size"]
    if FROZEN["model_sha256"] != sha256_file(MODEL_PATH):
        raise ValueError("Model differs from frozen tuning settings. This is a different experiment.")
    if FROZEN["tuning_clip_sha256"] == sha256_file(CLIP_PATH):
        raise ValueError("This is the same video used for tuning. Use a held-out test video.")

RULES.validate()
validate_site(SITE,CLIP_INFO['width'],CLIP_INFO['height'])
if 1/CLIP_INFO['fps'] > RULES.max_observation_gap_seconds:
    raise ValueError('Frame interval exceeds the observation-gap limit. Use suitable footage or revise the limit on tuning data.')
if not 0 < DETECTION_CONFIDENCE <= TRACKER_SETTINGS['track_low_thresh']:
    raise ValueError("Detection confidence must retain ByteTrack's low-confidence association candidates.")

def draw_site(frame, site):
    frame = frame.copy()
    poly = np.array(site['waiting_area'],dtype=np.int32)
    cv2.polylines(frame,[poly],True,(20,170,230),3)
    a,b = [tuple(map(int,p)) for p in site['crossing_line']]
    cv2.line(frame,a,b,(255,150,30),3)
    cv2.putText(frame,'A',a,cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,150,30),2)
    cv2.putText(frame,'B',b,cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,150,30),2)
    return frame

fig, axes = plt.subplots(1,3,figsize=(18,6))
for ax,index in zip(axes,[0,CLIP_INFO['frames']//2,CLIP_INFO['frames']-1]):
    frame = draw_site(read_video_frame(CLIP_PATH,index),SITE)
    ax.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
    ax.set_title(f"Frame {index}: {index/CLIP_INFO['fps']:.2f}s")
    ax.axis('off')
plt.tight_layout()
plt.show()
print("Check that the polygon and line stay on the SAME piece of ground in every frame.")


## 11. Confirm the capture and create a new saved run
Inspect the scene, including background landmarks. The three pictures cannot prove the camera
was stationary throughout, and a low pixel-difference score would not prove it either.
Set these confirmations only after checking your own video. In test mode confirm this is the
same physical camera setup as the frozen site; the frozen `recorded_on` is the site's calibration date.

In [ ]:
from datetime import datetime, timezone
from uuid import uuid4

# Current video is being used for an exploratory trial.
# Change these to True only after verifying the recording.
SITE_AND_CAMERA_CONFIRMED = False
CONSTANT_FRAME_RATE_CONFIRMED = False

# Reuse the actual recording date entered in section 10.
THIS_CLIP_RECORDED_ON = SITE.get("recorded_on")

CAPTURE_NOTES = (
    "Exploratory run; camera stability and constant frame rate "
    "have not yet been confirmed."
)

CAPTURE_VERIFIED = bool(
    SITE_AND_CAMERA_CONFIRMED
    and CONSTANT_FRAME_RATE_CONFIRMED
    and THIS_CLIP_RECORDED_ON
)

if not CAPTURE_VERIFIED:
    print(
        "TRIAL RUN: capture conditions are not fully verified. "
        "Use the results to inspect tracking, not as final "
        "validated performance."
    )

# Create a separate output folder for this run.
RUN_ID = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid4().hex[:6]
)

OUT_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "T4"
    / CLIP_PATH.stem
    / RUN_ID
)

OUT_DIR.mkdir(parents=True, exist_ok=False)

# Save the tracker settings used for this run.
TRACKER_PATH = OUT_DIR / "bytetrack.yaml"
TRACKER_PATH.write_text(
    yaml.safe_dump(TRACKER_SETTINGS),
    encoding="utf-8",
)

# Save the inputs, settings and recording notes.
MANIFEST = {
    "schema_version": 1,
    "run_id": RUN_ID,
    "phase": PHASE,
    "exploratory_run": not CAPTURE_VERIFIED,
    "capture_verified": CAPTURE_VERIFIED,
    "site_and_camera_confirmed": SITE_AND_CAMERA_CONFIRMED,
    "constant_frame_rate_confirmed": CONSTANT_FRAME_RATE_CONFIRMED,
    "clip": str(CLIP_PATH),
    "clip_sha256": sha256_file(CLIP_PATH),
    "model": str(MODEL_PATH),
    "model_sha256": sha256_file(MODEL_PATH),
    "ultralytics_version": ultralytics.__version__,
    "opencv_version": cv2.__version__,
    "python_version": sys.version,
    "clip_info": CLIP_INFO,
    "capture_date": THIS_CLIP_RECORDED_ON,
    "capture_notes": CAPTURE_NOTES,
    "site": SITE,
    "rules": asdict(RULES),
    "tracker_settings": TRACKER_SETTINGS,
    "detection_confidence": DETECTION_CONFIDENCE,
    "image_size": IMAGE_SIZE,
    "frozen_settings_path": (
        str(FROZEN_SETTINGS_PATH)
        if FROZEN_SETTINGS_PATH
        else None
    ),
}

save_json(OUT_DIR / "run_manifest.json", MANIFEST)
save_json(OUT_DIR / "site.json", SITE)
save_json(OUT_DIR / "rules.json", asdict(RULES))

# Save a frame showing the waiting area and crossing line.
marked_frame = draw_site(FIRST_FRAME, SITE)
saved = cv2.imwrite(
    str(OUT_DIR / "first_frame_marked.jpg"),
    marked_frame,
)

if not saved:
    raise RuntimeError("Could not save the marked first frame.")

print("Run created successfully.")
print("Video:", CLIP_PATH.name)
print("Results will be saved to:", OUT_DIR)


## 12. Run your trained detector and ByteTrack
This is the GPU step. Every frame is processed and the frame number gives its timestamp.
No stock model or drawn video is silently substituted. Tracks are saved immediately so later
rule tuning does not require repeating inference. Only eligible tracked IDs can be analysed;
missed people must be included in manual ground truth so the evaluation exposes them.

In [ ]:
if any(MANIFEST[key] != value for key,value in {
    'site':SITE,'rules':asdict(RULES),'tracker_settings':TRACKER_SETTINGS,
    'detection_confidence':DETECTION_CONFIDENCE,'image_size':IMAGE_SIZE}.items()):
    raise ValueError('Settings changed: create a new saved run before tracking.')
if MANIFEST['model_sha256'] != sha256_file(MODEL_PATH) or MANIFEST['clip_sha256'] != sha256_file(CLIP_PATH):
    raise ValueError('Video/model changed since this run was created. Create a new run.')
TRACKED = track_video(CLIP_PATH,MODEL_PATH,TRACKER_PATH,
                      confidence=DETECTION_CONFIDENCE,image_size=IMAGE_SIZE,
                      progress=lambda done,total: print(f"Tracked {done}/{total} frames"))
if not {'person_left','person_right'}.issubset(set(TRACKED['class_names'].values())):
    raise ValueError('This run is intended for your improved two-direction model; check best.pt.')
save_json(OUT_DIR/'tracks.json',TRACKED)
print(f"Saved {len(TRACKED['tracks'])} raw IDs. Processing speed: {TRACKED['processing_fps']:.1f} frames/s")
print('Processing throughput excludes upload, replay rendering and the future Gradio request overhead.')


## 13. Classify and save the T4 outputs
The totals describe **tracker identities**, not guaranteed unique human beings. Review
fragmentation before interpreting them as people counts. Direction comes from crossing movement;
the detector's majority direction label is retained as a diagnostic, not used to split IDs.

In [ ]:
# To resume a saved run after a runtime restart: rerun setup and definitions,
# restore SITE/RULES/OUT_DIR from its run_manifest.json, then load its tracks.json.
# TRACKED = json.loads((OUT_DIR/'tracks.json').read_text())
# TRACKED['tracks'] = {int(tid):obs for tid,obs in TRACKED['tracks'].items()}
if MANIFEST['rules'] != asdict(RULES) or MANIFEST['site'] != SITE:
    raise ValueError('Rules/site changed. Create a new saved run, or use the isolated tuning sweep below.')
RESULT = analyze_tracks(TRACKED['tracks'],SITE,RULES)
save_json(OUT_DIR/'summary.json',RESULT['summary'])
save_json(OUT_DIR/'analysis.json',RESULT)
save_csv(OUT_DIR/'people.csv',RESULT['people'], fields=[
    'track_id','eligible','outcome','waited','crossed','observed_wait_seconds','wait_episode_count',
    'duration_incomplete','first_crossing_seconds','first_seen_seconds','last_seen_seconds',
    'observations','majority_detector_class','flags'])
save_csv(OUT_DIR/'wait_episodes.csv',RESULT['wait_episodes'], fields=[
    'start_seconds','end_seconds','left_censored','seconds','qualified','end_reason',
    'right_censored','incomplete','track_id'])
save_csv(OUT_DIR/'crossings.csv',RESULT['crossings'], fields=[
    'seconds','confirmed_seconds','direction','screen_direction','track_id'])
timeline_rows = [dict(track_id=tid,**row) for tid,rows in RESULT['timelines'].items() for row in rows]
save_csv(OUT_DIR/'timeline.csv',timeline_rows,fields=[
    'track_id','frame','seconds','state','motion_body_heights_per_second','observed_wait_seconds','smooth_foot'])
display(pd.DataFrame(RESULT['people']))
print(json.dumps(RESULT['summary'],indent=2))
print('For duration_incomplete=True, read observed_wait_seconds as AT LEAST that duration.')


In [ ]:
# Leave as None to show everyone.
# Change to an ID number, such as 7, to inspect that person.
TARGET_ID = None

rows = []

for person in RESULT["people"]:
    if TARGET_ID is not None and person["track_id"] != TARGET_ID:
        continue

    if person["waited"]:
        prefix = "At least " if person["duration_incomplete"] else ""
        waiting_time = (
            f"{prefix}{person['observed_wait_seconds']:.2f} seconds"
        )
    else:
        waiting_time = "No qualifying wait observed"

    rows.append({
        "Person ID": person["track_id"],
        "Direction class": person["majority_detector_class"],
        "Waited": person["waited"],
        "Waiting time": waiting_time,
        "Number of waiting episodes": person["wait_episode_count"],
        "Crossed": person["crossed"],
        "Included in counts": person["eligible"],
    })

waiting_table = pd.DataFrame(rows)
display(waiting_table)

waiting_table.to_csv(
    OUT_DIR / "waiting_times_by_id.csv",
    index=False,
)

print("Saved to:", OUT_DIR / "waiting_times_by_id.csv")

## 14. Watch the annotated video and inspect the motion traces
Watch one real person at a time. Check stationary detections, ID changes, people passing each
other, zone boundaries and crossing confirmation. The overlay is an offline replay; its state
labels use preceding observations, but ID eligibility is determined from the completed track.
The state `possible_wait` is not counted until the minimum wait is reached.

In [ ]:
ANNOTATED_PATH = OUT_DIR/'annotated.mp4'
render_video(CLIP_PATH,TRACKED,RESULT,SITE,RULES,ANNOTATED_PATH)
import shutil
PLAYABLE_PATH = ANNOTATED_PATH
if shutil.which('ffmpeg'):
    PLAYABLE_PATH = OUT_DIR/'annotated_playable.mp4'
    subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(ANNOTATED_PATH),
                    '-an','-vf',"scale='min(1280,iw)':-2",'-c:v','libx264','-pix_fmt','yuv420p',
                    '-movflags','+faststart',str(PLAYABLE_PATH)],check=True)
if PLAYABLE_PATH.stat().st_size < 30*1024*1024:
    display(Video(str(PLAYABLE_PATH),embed=True,width=900))
else:
    print('Video saved; open it from Drive:',PLAYABLE_PATH)

fig, axes = plt.subplots(2,1,figsize=(12,7),sharex=True)
for tid,rows in RESULT['timelines'].items():
    t=[r['seconds'] for r in rows]
    axes[0].plot(t,[r['motion_body_heights_per_second'] for r in rows],label=f'ID {tid}')
    axes[1].plot(t,[r['observed_wait_seconds'] for r in rows])
axes[0].axhline(RULES.still_body_heights_per_second,color='black',linestyle='--')
axes[0].set_ylabel('Motion (body heights/s)')
axes[1].axhline(RULES.min_wait_seconds,color='black',linestyle='--')
axes[1].set_ylabel('Current observed stillness (s)')
axes[1].set_xlabel('Video time (s)')
if len(RESULT['people']) <= 12 and RESULT['people']:
    axes[0].legend(loc='upper right')
plt.tight_layout()
plt.savefig(OUT_DIR/'motion_and_waits.png',dpi=150)
plt.show()


## 15. Create your manual evaluation files
Annotate **all relevant real people by watching the original video**, including people the
model missed. Assign human IDs such as P01, P02 yourself, not from the tracker.
Do not copy predicted waiting times into ground truth. Use the same operational definition,
including the minimum waiting duration. `wait_seconds` is total qualifying waiting time for
that person; leave it empty for an incomplete wait, and set `duration_complete=false`.

Ground truth columns: `human_id, waited, crossed, wait_seconds, duration_complete, notes`.
Boolean fields use `true` or `false`. Use notes for uncertainty and boundary cases.

After independent annotation, watch the replay and fill `id_mapping.csv`:
`human_id, track_id, notes`. Use multiple rows if a real person acquired several IDs.
Use a blank track_id for a missed person. Leave false-positive IDs unmapped. A track must
not be assigned to two people: a suspected ID swap belongs in the failure review.
Ground truth persists per clip; mappings are specific to this run's tracking IDs.

In [ ]:
LABEL_DIR = PROJECT_ROOT/'data'/'t4_labels'/CLIP_PATH.stem
LABEL_DIR.mkdir(parents=True,exist_ok=True)
TRUTH_CSV = LABEL_DIR/'ground_truth.csv'
MAPPING_CSV = OUT_DIR/'id_mapping.csv'
context_path = LABEL_DIR/'label_context.json'
if context_path.exists():
    if json.loads(context_path.read_text())['clip_sha256'] != MANIFEST['clip_sha256']:
        raise ValueError('These labels belong to different video bytes. Use a different clip name/label directory.')
else:
    save_json(context_path,{'clip_sha256':MANIFEST['clip_sha256'],'clip':str(CLIP_PATH)})
if not TRUTH_CSV.exists():
    save_csv(TRUTH_CSV,[],fields=['human_id','waited','crossed','wait_seconds','duration_complete','notes'])
if not MAPPING_CSV.exists():
    save_csv(MAPPING_CSV,[],fields=['human_id','track_id','notes'])
FAILURE_CSV = OUT_DIR/'failure_review.csv'
if not FAILURE_CSV.exists():
    save_csv(FAILURE_CSV,[],fields=['human_id','track_id','start_seconds','end_seconds','stage','description','evidence_file'])
print('Fill independently:',TRUTH_CSV)
print('Then map real people to replay IDs:',MAPPING_CSV)
print('Record detector/tracker/rule/capture failures:',FAILURE_CSV)


## 16. Evaluate only after filling the files
Count errors use the raw predicted ID totals, so fragmentation remains visible. Person-grouped
classification is reported separately after manual mapping. Missed people remain in the
denominator and unmatched eligible IDs contribute false positives. Duration MAE is calculated
only for complete, correctly detected waits with one mapped eligible track; coverage and
excluded people are reported alongside it. A low MAE with poor coverage is not a strong result.
`null` means a metric is undefined, not zero error. An empty ground-truth file cannot produce a score.

In [ ]:
RUN_EVALUATION = False                       # Set True after genuine manual annotation
EVALUATION = None
if RUN_EVALUATION:
    with open(TRUTH_CSV,newline='',encoding='utf-8-sig') as f:
        TRUTH_ROWS = list(csv.DictReader(f))
    with open(MAPPING_CSV,newline='',encoding='utf-8-sig') as f:
        MAPPING_ROWS = list(csv.DictReader(f))
    EVALUATION = evaluate_people(RESULT,TRUTH_ROWS,MAPPING_ROWS)
    save_json(OUT_DIR/'evaluation.json',EVALUATION)
    save_csv(OUT_DIR/'evaluation_people.csv',EVALUATION['people'])
    save_json(OUT_DIR/'evaluation_provenance.json',{
        'truth_file':str(TRUTH_CSV),'truth_sha256':sha256_file(TRUTH_CSV),
        'mapping_file':str(MAPPING_CSV),'mapping_sha256':sha256_file(MAPPING_CSV),
        'phase':PHASE,'run_id':RUN_ID})
    display(pd.DataFrame(EVALUATION['people']))
    print(json.dumps(EVALUATION['metrics'],indent=2))
else:
    print('Evaluation skipped. No measured accuracy is claimed.')


## 17. Optional: compare waiting rules on TUNING data only
This reuses the same tracks; it does not test alternative detector/tracker settings.
Compare duration coverage and failure videos as well as scores. Test tracker changes in separate
saved runs, since different settings can change IDs and require new mappings.
Choose thresholds on tuning clips representing several cases; never pick them using test results.
The table does not automatically declare a winner.

In [ ]:
RUN_RULE_SWEEP = False
if RUN_RULE_SWEEP:
    if PHASE != 'tuning':
        raise ValueError('Rule sweeps are disabled on final test clips.')
    if EVALUATION is None:
        raise ValueError('Run the manually annotated evaluation first.')
    comparisons = []
    for still_threshold in (0.03,0.06,0.10):
        # Keep the definition of waiting fixed: changing minimum seconds would change the human labels too.
        trial_rules = replace(RULES,still_body_heights_per_second=still_threshold)
        trial = analyze_tracks(TRACKED['tracks'],SITE,trial_rules)
        metrics = evaluate_people(trial,TRUTH_ROWS,MAPPING_ROWS)['metrics']
        comparisons.append({'min_wait_seconds':RULES.min_wait_seconds,'still_threshold':still_threshold,
            **{key:metrics[key] for key in ('person_grouped_classification_accuracy',
              'waited_raw_count_absolute_error','crossed_raw_count_absolute_error',
              'duration_mae_seconds','duration_evaluated_people')}})
    display(pd.DataFrame(comparisons))
    save_csv(OUT_DIR/'tuning_rule_comparison.csv',comparisons)


## 18. Freeze settings before held-out testing
After choosing settings from your tuning evidence, rerun analysis/evaluation with those
settings, then explicitly freeze this configuration. Preserve the operational definition
between tuning labels and test labels. Use the same camera geometry and different video bytes.
For several test clips, repeat the run for each and retain every result rather than only the best.

In [ ]:
FREEZE_THIS_CONFIGURATION = False
if FREEZE_THIS_CONFIGURATION:
    if PHASE != 'tuning' or EVALUATION is None:
        raise ValueError('Freeze only an evaluated tuning run.')
    if any(MANIFEST[key] != value for key,value in {
        'site':SITE,'rules':asdict(RULES),'tracker_settings':TRACKER_SETTINGS,
        'detection_confidence':DETECTION_CONFIDENCE,'image_size':IMAGE_SIZE}.items()):
        raise ValueError('Settings changed after this saved run. Run and evaluate the chosen settings first.')
    frozen = {'schema_version':1,'site':SITE,'rules':asdict(RULES),
              'tracker_settings':TRACKER_SETTINGS,'detection_confidence':DETECTION_CONFIDENCE,
              'image_size':IMAGE_SIZE,'model_sha256':MANIFEST['model_sha256'],
              'tuning_clip_sha256':MANIFEST['clip_sha256'],'source_run_id':RUN_ID,
              'ultralytics_version':ULTRALYTICS_VERSION}
    frozen_path = PROJECT_ROOT/'artifacts'/'T4'/f'frozen_settings_{RUN_ID}.json'
    save_json(frozen_path,frozen)
    print('For the held-out run, use PHASE="test" and this FROZEN_SETTINGS_PATH:',frozen_path)


## 19. Save failure examples and list your evidence
Choose times after inspecting the videos; these are not automatically proven failures.
For each example, enter what happened in failure_review.csv and identify its stage: detector
(missed box), tracker (ID split/swap), rule (false stillness/zone decision), or capture (camera/occlusion).

In [ ]:
FAILURE_EXAMPLE_SECONDS = []                 # e.g. actual inspected video times, not guessed examples
for second in FAILURE_EXAMPLE_SECONDS:
    if not 0 <= second < CLIP_INFO['duration_seconds']:
        raise ValueError('Failure example lies outside this video.')
    index = round(second*CLIP_INFO['fps'])
    image = read_video_frame(CLIP_PATH,index)
    cv2.imwrite(str(OUT_DIR/f'original_at_{second:.2f}s.jpg'),image)
    annotated = read_video_frame(ANNOTATED_PATH,index)
    cv2.imwrite(str(OUT_DIR/f'annotated_at_{second:.2f}s.jpg'),annotated)
print('Saved evidence files:')
for path in sorted(OUT_DIR.iterdir()):
    print(path.name)


## Before the separate Gradio demo notebook
The companion `t4_engine.py` contains the same functions as this notebook. Upload that helper
beside your later demo notebook to reuse `track_video`, `analyze_tracks` and `render_video`.
The demo should load your frozen model/settings, accept a video from the **same camera site**,
or require new site marking for a new camera. Each request needs a fresh tracker and its own
output directory. Show counts, observed durations, incomplete-duration flags and annotated replay.

Your next real-work checkpoints:
- Hand-labelled tuning and held-out videos, including a no-person clip.
- A walking-through case, a short pause, a qualifying wait then crossing, an outside-zone stop,
  overlapping people, a slow walker, and waits already in progress at the clip boundaries.
- Measured count/classification results, duration error with coverage, and honest failure examples.
- Confirm stationary people are detected: the directional training convention can exclude
  undeterminable poses. A tracking threshold cannot recover a person the detector never finds.
- Keep your baseline image deployment available; rehearse the separate T4 demo and record a backup.

The constructed-track checks distributed with this notebook verify rule logic only.
They do not measure your detector, tracker or accuracy on real people. Your report must explain
your own decisions and observations; do not treat the notebook text as a report to submit.